# Autobot Exp 7: 4-Stage Sequential SWE Pipeline with Isolated Contexts
## Architecture: Localizer -> Reproducer -> Patcher -> Finalizer
### SOTA Principles: Staged isolation (`include_contents: none`), unique anchors, guarded in-place fallback, guaranteed submission.


In [ ]:
import os, sys, shutil, zipfile, hashlib, json
from pathlib import Path

WORKING_DIR = Path('/kaggle/working').resolve()
AGENT_DIR = WORKING_DIR / 'submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
AGENT_DIR.mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'configs').mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'prompts').mkdir(parents=True, exist_ok=True)
(AGENT_DIR / 'sub_agents').mkdir(parents=True, exist_ok=True)

print(f'Initialized build directory at: {AGENT_DIR}')


In [ ]:
# 1. Configs & Budgets
SAMPLING_YAML = 'temperature: 0.15\ntop_p: 0.92\ntop_k: 40\nmax_output_tokens: 16384\nseed: 42\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n'
(AGENT_DIR / 'configs' / 'sampling.yaml').write_text(SAMPLING_YAML.strip() + '\n', encoding='utf-8')
EVAL_YAML = 'evaluation:\n  timeout_seconds: 180\n  max_tool_calls: 60\n  max_time_minutes: 8.0\n  max_turns: 80\n'
(AGENT_DIR / 'eval_config.yaml').write_text(EVAL_YAML.strip() + '\n', encoding='utf-8')

# 2. Prompts
LOCALIZER_MD = 'You are `localizer`, the code exploration specialist in `/workspace`.\nYour single mission is to locate the exact source files, function symbols, and root cause of the issue described below.\n\nTask Problem Statement:\n{problem_description}\n\n## Tools Available\n- `read_file(filepath, start_line, end_line)`: Inspect files with focused line ranges (<= 150 lines).\n- `search_similar_code(query)`: Query semantic code graph using a single exact symbol name (e.g. `HTTPConnection` or `parse_header`), NOT natural language.\n- `get_code_neighbors(node)`: Inspect callers/callees around a known symbol.\n- `get_code_subgraph(nodes)`: Inspect graph connectivity between symbols.\n- `run_command(command)`: Use strictly for fast discovery (`rg -n "<symbol>"`, `find . -name "*.py"`). Limit command output with `2>&1 | tail -n 30`.\n\n## Exploration Budget & Rules\n1. Make at most 10–12 tool calls total. Do not explore indefinitely.\n2. Search for exact symbols, error messages, and parameters from the issue.\n3. Once the target file and lines are confirmed by reading the code, STOP calling tools immediately.\n4. Output ONLY the following structured card verbatim (no conversational intro or outro):\n\nFILES: <path>:<start>-<end> for each region to change (max 3)\nSYMBOLS: <function/class names involved>\nCURRENT: <one sentence: what the code does now>\nEXPECTED: <one sentence: what the issue says it must do; copy exact names, messages, and exception types from the issue verbatim>\nROOT CAUSE: <one or two sentences explaining the bug>\nPLAN: <at most 3 numbered edits, each naming file + function>\nTEST HINT: <existing test file under tests/ most likely to cover this, or "none found">\n'
(AGENT_DIR / 'prompts' / 'localizer.md').write_text(LOCALIZER_MD.strip() + '\n', encoding='utf-8')
REPRODUCER_MD = 'You are `reproducer`, the bug verification and reproduction specialist in `/workspace`.\nYour mission is to establish a fast, reliable verification command that fails on the current unpatched code.\n\nTask Problem Statement:\n{problem_description}\n\nDiagnostic Findings from Localizer:\n{bug_card}\n\n## Tools Available\n- `run_command(command)`: Execute bash commands. Always append `2>&1 | tail -n 25` to keep context clean.\n- `read_file(filepath, start_line, end_line)`: Inspect existing tests under `tests/`.\n\n## Reproduction Rules\n1. Never leave scratch files in `/workspace`! All temporary scripts must live in `/tmp` (e.g. `/tmp/repro.py`).\n2. Method A (Existing Unit Test):\n   Check the `TEST HINT` from the bug card. Run ONLY the targeted test:\n   `PYTHONDONTWRITEBYTECODE=1 timeout 60 python3 -m pytest -q -x -k "<symbol>" <test_file> 2>&1 | tail -n 25`\n3. Method B (Inline Heredoc Check):\n   If no existing test is readily isolated, execute a minimal inline python snippet reproducing the bug:\n   `cd /workspace && PYTHONDONTWRITEBYTECODE=1 timeout 60 python3 - <<\'EOF\' 2>&1 | tail -n 20`\n4. Confirm that the command FAILS on the current code. If it passes, the test does not capture the bug.\n5. Spend at most 6–8 tool calls. Conclude with ONLY this structured card:\n\nREPRO_COMMAND: <exact bash command to run>\nFAILURE_OUTPUT: <at most 5 lines of the failure traceback or assert error>\nSUCCESS_CRITERION: <exit code 0 or expected output string>\n'
(AGENT_DIR / 'prompts' / 'reproducer.md').write_text(REPRODUCER_MD.strip() + '\n', encoding='utf-8')
PATCHER_MD = 'You are `patcher`, the surgical software repair engineer in `/workspace`.\nYour mission is to implement a clean, minimal fix in the library code that resolves the defect and passes the reproduction check.\n\nTask Problem Statement:\n{problem_description}\n\nDiagnostic Plan:\n{bug_card}\n\nReproduction Command:\n{repro_card}\n\n## Tools Available\n- `read_file(filepath, start_line, end_line)`: Inspect target source regions.\n- `edit_file(filepath, old_string, new_string)`: Replace exact text blocks.\n- `run_command(command)`: Execute syntax checks, tests, and diff inspections. Append `2>&1 | tail -n 25`.\n\n## Absolute Anti-Tampering Constraint\n- **DO NOT MODIFY OR DELETE ANY TEST FILES** (`tests/`, `test_*.py`, `*_test.py`) or test configs (`conftest.py`, `pytest.ini`, `setup.cfg`).\n- The evaluation harness discards any test modifications before scoring. All fixes must be in the implementation source code.\n\n## Edit Rules (Follow Strictly to Prevent FileEditError)\n1. **Read Before Edit**: Always call `read_file` on the target region first. Copy `old_string` character-for-character with line-number prefixes stripped.\n2. **Unique Anchors**: `old_string` must be unique in the file. Include 3–4 unchanged lines above and below the change, or for functions under 40 lines, replace the entire `def` function block.\n3. **Exact Indentation**: `new_string` must preserve the exact indentation level of the code it replaces.\n4. **Fallback In-Place Replacement**: If `edit_file` fails twice on a location, use a guarded in-place Python replacement via `run_command`:\n   ```bash\n   cd /workspace && python3 - <<\'EOF\'\n   import pathlib\n   p = pathlib.Path("<filepath>")\n   s = p.read_text(encoding="utf-8")\n   old = """<exact_old_code>"""\n   new = """<exact_new_code>"""\n   assert s.count(old) == 1, f"anchor matched {s.count(old)} times"\n   p.write_text(s.replace(old, new, 1), encoding="utf-8")\n   print("REPLACE_OK")\n   EOF\n   ```\n\n## Verification Procedure\n1. Verify Python syntax: `run_command("python3 -m py_compile <modified_file>")`.\n2. Run the `REPRO_COMMAND` from the reproducer card. Confirm that it now PASSES (exit code 0).\n3. Inspect diff: `run_command("git diff -- <modified_file>")`. Ensure only the necessary lines changed.\n4. Output a concise summary of the applied fix and test verification result.\n'
(AGENT_DIR / 'prompts' / 'patcher.md').write_text(PATCHER_MD.strip() + '\n', encoding='utf-8')
FINALIZER_MD = 'You are `finalizer`, the quality assurance and submission officer in `/workspace`.\nYour single mission is to ensure the repository working tree is clean, verified, and submitted via `submit_patch()`.\n\nTask Problem Statement:\n{problem_description}\n\nPatch Summary:\n{patch_note}\n\n## Tools Available\n- `run_command(command)`: Execute status checks and cleanup.\n- `submit_patch()`: Final submission action.\n\n## Finalization Checklist (Execute Sequentially)\n1. **Clean Scratch Files**:\n   Run `git status --porcelain`. If any untracked scratch files were left in `/workspace`, delete them immediately with `rm -f <file>`.\n2. **Anti-Tampering Enforcement**:\n   Run `git diff --name-only`.\n   If any files under `tests/`, `test/`, `testing/`, or config files (`conftest.py`, `pytest.ini`, `setup.cfg`, `tox.ini`) were modified, revert them immediately:\n   `git checkout -- <test_file>`\n   Container B wipes any test changes, so test modifications must not appear in the patch.\n3. **Diff Sanity Check**:\n   Run `git diff --check` and `git diff --stat` to ensure the patch is clean, well-formatted, and modifies only library source code.\n4. **Call `submit_patch()`**:\n   Call `submit_patch()` as your definitive action. It stages the working tree and records the final evaluation diff.\n5. Conclude with a brief 2-sentence confirmation of the submitted files and patch status.\n'
(AGENT_DIR / 'prompts' / 'finalizer.md').write_text(FINALIZER_MD.strip() + '\n', encoding='utf-8')

# 3. Sub-Agent Declarations
LOCALIZER_YAML = 'name: localizer\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Code exploration specialist that identifies the exact files, root cause, and test target.\ninstruction: !include ../prompts/localizer.md\ngenerate_content_config: !include ../configs/sampling.yaml\ntools:\n  - run_command\n  - read_file\n  - search_similar_code\n  - get_code_neighbors\n  - get_code_subgraph\noutput_key: bug_card\n'
(AGENT_DIR / 'sub_agents' / 'localizer.yaml').write_text(LOCALIZER_YAML.strip() + '\n', encoding='utf-8')
REPRODUCER_YAML = 'name: reproducer\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Bug verification specialist that creates a minimal reproduction command that fails on unpatched code.\ninclude_contents: none\ninstruction: !include ../prompts/reproducer.md\ngenerate_content_config: !include ../configs/sampling.yaml\ntools:\n  - run_command\n  - read_file\noutput_key: repro_card\n'
(AGENT_DIR / 'sub_agents' / 'reproducer.yaml').write_text(REPRODUCER_YAML.strip() + '\n', encoding='utf-8')
PATCHER_YAML = 'name: patcher\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Surgical patch engineer that implements and verifies the code change.\ninclude_contents: none\ninstruction: !include ../prompts/patcher.md\ngenerate_content_config: !include ../configs/sampling.yaml\ntools:\n  - read_file\n  - edit_file\n  - run_command\noutput_key: patch_note\n'
(AGENT_DIR / 'sub_agents' / 'patcher.yaml').write_text(PATCHER_YAML.strip() + '\n', encoding='utf-8')
FINALIZER_YAML = 'name: finalizer\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Quality assurance officer that cleans scratch files, enforces test anti-tampering, and calls submit_patch.\ninclude_contents: none\ninstruction: !include ../prompts/finalizer.md\ngenerate_content_config: !include ../configs/sampling.yaml\ntools:\n  - run_command\n  - submit_patch\n'
(AGENT_DIR / 'sub_agents' / 'finalizer.yaml').write_text(FINALIZER_YAML.strip() + '\n', encoding='utf-8')

# 4. Root Sequential Pipeline
AGENT_YAML = 'name: autobot_swe_pipeline\nagent_class: SequentialAgent\ndescription: 4-stage sequential software engineering pipeline with isolated contexts for surgical repository repair.\nsub_agents:\n  - config_path: sub_agents/localizer.yaml\n  - config_path: sub_agents/reproducer.yaml\n  - config_path: sub_agents/patcher.yaml\n  - config_path: sub_agents/finalizer.yaml\n'
(AGENT_DIR / 'agent.yaml').write_text(AGENT_YAML.strip() + '\n', encoding='utf-8')

print('Wrote all 11 pipeline configuration and prompt files successfully.')


In [ ]:
# 5. Package submission.zip with deterministic timestamps
ZIP_PATH = WORKING_DIR / 'submission.zip'
ZIP_PATH.unlink(missing_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(p for p in AGENT_DIR.rglob('*') if p.is_file()):
        rel_path = str(p.relative_to(AGENT_DIR)).replace('\\', '/')
        info = zipfile.ZipInfo(rel_path, date_time=(1980, 1, 1, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        info.external_attr = 0o644 << 16
        zf.writestr(info, p.read_bytes())

size_kb = ZIP_PATH.stat().st_size / 1024.0
print(f'Successfully created {ZIP_PATH.name} ({size_kb:.2f} KB)')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    for name in zf.namelist():
        print(f'  - {name}')


In [ ]:
# 6. Rigorous validation against swegemma and adk-submission limits
import yaml

class SafeIncludeLoader(yaml.SafeLoader):
    pass
SafeIncludeLoader.add_constructor('!include', lambda loader, node: loader.construct_scalar(node))

with open(AGENT_DIR / 'agent.yaml', 'r', encoding='utf-8') as f:
    root_cfg = yaml.load(f, Loader=SafeIncludeLoader)
assert root_cfg['agent_class'] == 'SequentialAgent', 'Root must be SequentialAgent'

models = set()
for sub in root_cfg['sub_agents']:
    sub_p = (AGENT_DIR / sub['config_path']).resolve()
    assert sub_p.exists(), f'Missing sub-agent file: {sub["config_path"]}'
    with open(sub_p, 'r', encoding='utf-8') as f:
        sub_c = yaml.load(f, Loader=SafeIncludeLoader)
    models.add(sub_c['model'])

assert models == {'gemma-4-31b-it-qat-w4a16-ct'}, f'Single base model violation: {models}'
assert ZIP_PATH.stat().st_size < 3 * 1024 * 1024 * 1024, 'Archive exceeds 3 GiB'

print('\n======================================================')
print('SUCCESS: EXP 7 SEQUENTIAL PIPELINE VALIDATION PASSED!')
print(f'Submission: {ZIP_PATH} ({size_kb:.2f} KB) ready for scoring.')
print('======================================================')
